# Project Management Dashboard

**Authors:** [Your Names Here]

**Course:** CS 188

**Date:** December 2025

## Abstract

[Provide a brief summary of the project, its goals, key contributions, and main findings. This should be 150-250 words.]

## 1. Introduction

[Introduce the problem you're solving, why it matters, and what your solution is. Include:
- Problem statement
- Motivation
- Key contributions
- Paper organization]

## 2. Background & Related Work

This section establishes the foundation for our work and justifies why a new approach combining formal systems engineering with practical dependency management was necessary.

### Systems Engineering Approaches: MBSE, ConOps, and Semantic Modeling

**What exists**: Model-Based Systems Engineering (MBSE) is a prominent practice in System Engineering, motivated by the need to formalize models in order to avoid the ambiguity of natural language. As such, capturing the system in a living, formal document is a vital step of the systems engineering process. One such way of doing this is by first starting with a living ConOps document. The main benefit of these documents is that they can describe a system from the stakeholders perspectives, including the requirements, the related capabilities, and the different scenarios and entities involved. Further, all of this is done without implementation details, allowing for flexibility in the implementation while maintaining the concept of building the right system. 

Once a ConOps document has been established, one tool that can be used to convert this information from a document format to a true formal model is the Ontological Modeling Language (OML). OML provides a rigorous foundation for capturing the information that describes system concepts, their properties, and relationships. OML organizes knowledge into vocabularies (defining conceptual schemas) and descriptions (creating specific instances), enabling reusability and consistent modeling patterns. Lastly, once the system has been formalized with an OML model, system engineers may want to query the model, to be able to ask interesting questions about the system. One existing tool to accomplish this is SPARQL, an SQL-like query language, which provides powerful pattern-matching capabilities for querying graph-structured data. The results of the queries, stored as json objects, can then be used for other tasks such as visualization. 

**What are their limits**: Traditional MBSE tools are not designed for software-specific challenges like multi-repository dependency management, and therefore do not natively integrate with platforms like GitHub. Further, while these tools may allow artifacts like a specific package release as an entity, there is no genuine understanding of what that artifact actually is. 

As previously mentioned, ConOps documents do not contain specific details about the implementation. So, for the software side of the project management system, the ConOps document wouldn’t have any mechanism linking it to the specific code or repository. Further, because the ConOps document lives separate from our actual technical artifacts, changes in one area must also be made in the other, which can create some redundancy in the work being done.

Lastly, as previously mentioned, the results of the SPARQL queries are JSON objects. While these objects may be easily machine parsable, they are not the most readable for humans. This is especially true when the query is complex or returns lots of results. Therefore, in order for it to be effectively used at the scale of the fireforce6 project, it requires supplemental tools to help portray the results of the query.

**How we are different**: Our approach applies the MBSE rigor to the entirety of the project management environment system, including the software side. We maintain the formalism needed in a system like this with OML, which helps us bridge the gap between formal modeling and practical implementation. Further, the entirety of our OML model lives within our project management environment repository, which helps to ensure that our model remains central in our system and evolves alongside the actual implementation.

We first started by describing our system in its entirety within a ConOps document. Next, we formalized our ConOps document directly in the OML model, creating machine-readable representations of stakeholders, requirements, missions, capabilities, processes, and scenarios. In order to do so, we leveraged the Sierra Method repository, which allowed us to include these crucial building blocks in our own code. This allowed for traceability throughout the entire system, allowing us to objectively determine which processes implement a given capability, which capabilities satisfy a requirement, and which requirements a stakeholder expressed. 

Once our OML model was complete and pushed to our repository, we began to ask questions about our system with SPARQL. Our SPARQL queries extract project management insights (requirements traceability, capability dependencies, process allocations) from OML models and feed them into Jupyter notebooks for visualization. This addresses one of the major limitations of SPARQL previously mentioned, which is that the results are not easily human readable. In other words, we built and analyzed our system using a pipeline, starting from our ConOps documentation, to an OML Model, to SPARQL queries, to visualization with Python.

### Repository Management and Dependency Visualization

**What exists**: GitHub and its ecosystem provide tools for coordinating multi-repository projects. GitHub also provides pre-made “GitHub Apps”, installable to individual repositories or at the organizational level, along with the ability to custom-make these apps. At the package level, tools like Maven manage dependencies within individual projects using specification files. In order to keep related constituent systems together, GitHub allows their repositories to all be placed within a single organization, but this simply indicates that these subsystems are somehow related, providing no information about how they depend on each other.

In terms of software dependency visualization, different tools visualize software dependencies at different levels. Package managers provide tree views, showing transitive dependencies within a project. Architecture documentation tools like PlantUML and Mermaid can also be used manually to create diagrams.

**What are their limits**: Many existing dependency management tools operate primarily at the individual repository level and lack organization-wide visibility. Further, their ability to effectively visualize these dependencies is questionable. However, the more critical limitation is that existing tools don’t provide traceability from stakeholder needs through requirements to capabilities and implementation. Instead, they solely focus on technical dependencies without the systems engineering context of why certain architecture decisions were made or which stakeholder needs they serve. Another approach is to attempt to manually track the dependencies within a system. However, these methods are extremely prone to becoming stale, and don’t provide any way to programmatically analyze. 

The bottom line is that existing visualization tools operate at either too low a level (code/package dependencies within a single project) or require manual creation (architecture diagrams that can become outdated). They don't automatically discover dependencies across multiple repositories, don't connect visualizations to formal system models, and don't provide multiple stakeholder-specific views. A developer may need detailed technical dependency information while a project manager needs a high-level overview, and existing tools don't serve both audiences from a single underlying model.

**How we are different**: Our system is different because it provides organization-level dependency tracking and visualization across all six fireforce6 repositories while maintaining formal traceability back to stakeholder requirements. Rather than just showing technical dependencies, our OML model connects the dependency architecture to specific capabilities and requirements expressed by the stakeholders. In other words, we took a systems engineering perspective that transforms dependency management from a purely technical task into a traceable design decision that satisfies stakeholder needs.

We maintain the multi-repository structure that allows each subsystem to work independently while adding a formal systems engineering layer that doesn't exist in traditional multi-repo approaches. Our GitHub App provides automated data collection ensuring that the system always reflects current repository state rather than relying on manual updates. The OML model enables programmatic querying through SPARQL, allowing us to generate views like "all capabilities that depend on c1-authorization" or "all processes allocated to the dashboard entity" that would be difficult or impossible with static documentation.

Our visualizations are generated with python using the results fetched using the GitHub App. We provide visualizations which are aware of not only dependencies, but also other aspects of the constituent systems such as release versions and hours logged. Further, the color-coded component of our graph allows fellow system engineers to understand the state of the fireforce6 organization at a glance. Lastly, the most important piece is that our visualizations are not manually created, but are instead programmatically generated and make use of the GitHub app.


## 3. System Architecture & Design

[Describe the architecture and design of your system. Include diagrams and explanations of:
- Overall system architecture
- OML vocabulary design
- Data flow
- Key components]

In [1]:
# Import utilities for generating diagrams
from utilities import *

### 3.1 Stakeholders

In [2]:
df = dataframe("stakeholders.json")
df

,stakeholder_id,stakeholder_desc
0,developer,Developers working in each subproject who need...
1,project-manager,Project manager who oversees all subprojects a...


### 3.2 Requirements

In [3]:
df = dataframe("requirements.json")
requirements = df.drop_duplicates(subset=['req_id'])[['req_id', 'req_desc']]
requirements = requirements.rename(columns={"req_id": "Requirement ID", "req_desc": "Description"})

style = requirements.style.hide(axis="index").set_properties(**{'text-align': 'left', 'font-size': '11pt'})
style.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])

Requirement ID,Description
r1-visual-overview,The dashboard must provide a visual overview of the dependencies and current releases for each sub-project.
r2-file-compatibility,The dashboard must be compatible with dependency specification files in each subproject's repository.
r3-up-to-date,The dashboard must have up-to-date versions and dependencies displayed in its graph.


### 3.3 Capabilities

In [4]:
df = dataframe("capabilities.json")
capabilities = df.drop_duplicates(subset=['cap_id'])[['cap_id', 'cap_desc']]
capabilities = capabilities.rename(columns={"cap_id": "Capability ID", "cap_desc": "Description"})

style = capabilities.style.hide(axis="index").set_properties(**{'text-align': 'left', 'font-size': '11pt'})
style.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])

Capability ID,Description
c1-authorization,Authorization: Server should gain authorization from GitHub App to access repository data.
c2-fetch-capability,Fetch Capability: Server should be able to fetch from each of the subproject repositories to grab their current version and dependencies.
c3-dashboard-display,Dashboard Display: Dashboard should be able to take fetched dependencies and display them in a graph format.
m1-at-a-glance-overview,"At a glance Overview: Needs clear visuals, displaying project connections and dependencies across the entire organization."
m2-easy-integration,Easy Integration: Needs the ability to access files in subproject repositories without complex setup or manual configuration.


### 3.4 Capability Dependencies

In [5]:
df = dataframe("capability-dependencies.json")
capabilities1 = todict(df, 'cap1_id', 'cap1_desc')
capabilities2 = todict(df, 'cap2_id', 'cap2_desc')
dependencies = tolists(df, 'cap1_id', 'cap2_id')

# Build PlantUML
uml = "@startuml\n"

all_caps = union(capabilities1, capabilities2)
for key, value in all_caps.items():
    alias = key.replace('-', '_')
    uml += 'object "'+key+'" as '+alias+' <<capability>>\n'

for row in dependencies:
    from_alias = row[0].replace('-', '_')
    to_alias = row[1].replace('-', '_')
    uml += from_alias+' --> '+to_alias+'\n'

uml += "@enduml\n"

diagram(uml)

![Alt text](http://www.plantuml.com/plantuml/img/TP313eCW44JlV0NnlWVrQeX_GfQ51Xeg1jP3-_LDhP5OxN5dTbucCnR6pCiZYcJkZbWsrC7DCNaWdD646FZPI2oIEhtgkkfo6EgXL4NqOB5uap1RiA7C4JT6htT3RyPVI0kui4yvl913chw0LX_4t_1LIG1roedB9kldcI16DzdFH6y=)

## 4. Demonstration & Evaluation

[Demonstrate your system and evaluate its effectiveness. Include:
- Screenshots or outputs from the web dashboard
- Example queries and results
- Evaluation metrics
- User feedback (if applicable)]

### 4.1 Requirements to Capabilities Traceability

In [6]:
df = dataframe("capabilities.json")

cap_df = df[df['cap_id'].str.startswith('c', na=False)]
capabilities_dict = todict(cap_df, 'cap_id', 'cap_desc')

req_ids = cap_df[cap_df['req_id'].notna()]['req_id'].unique()
requirements_dict = {req_id: req_id for req_id in req_ids}

derivations = tolists(cap_df, 'cap_id', 'req_id')

uml = "@startuml\n"

for key in requirements_dict.keys():
    if pd.notna(key):
        alias = key.replace('-', '_')
        uml += 'object "'+key+'" as '+alias+' <<requirement>>\n'

for key, value in capabilities_dict.items():
    if pd.notna(value):
        alias = key.replace('-', '_')
        uml += 'object "'+key+'" as '+alias+' <<capability>>\n'

for row in derivations:
    if pd.notna(row[0]) and pd.notna(row[1]):
        cap_alias = row[0].replace('-', '_')
        req_alias = row[1].replace('-', '_')
        uml += cap_alias+' ..> '+req_alias+' : isDerivedFrom\n'

uml += "@enduml\n"

diagram(uml)

![Alt text](http://www.plantuml.com/plantuml/img/hPB1RiCW44Jl-GgKEv4StwB8og7gRwmmREIjiA6mk6g_lZYaTHt7jrvdz4OpB9V8Ad3gxSwrRwXPxCDHNicYrAxtYgabI_ov5ogAS8J9WOKZpkU0xua2zZXmqymvjKnUifD6CHQ-XkI17KpADbV9aM9ILheHmqZuKU0AYWm_ycQ2hgHAtBb0Nxcc6swyhc0XPbehhyg8lt2UZwmxERk5p-Cd7MPzCwBEcsFNEXMCD1IzLVUC6duDk1kF8QkUXqEpyV8dGV450ureNeFhTyyg_WEzkDgAUZRfXgtlzSXUS33Jp_i6)

## 5. Conclusion & Future Work

[Summarize your contributions and discuss future directions. Include:
- Summary of what was accomplished
- Limitations of current approach
- Future improvements
- Lessons learned]

## References

1. [Reference 1]
2. [Reference 2]
3. [Reference 3]